In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text

c:\Users\LOQ\anaconda3\envs\tf_final\lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [3]:
df = pd.read_csv(r'D:\Intelligent-Support-Ticket\Project Implementation\Data\clean_data.csv')

In [4]:
df.head()

,created_at,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,platform,region,date
0,2024-01-31T05:14:27,individual,email,data_export,account_access,low,resolved,standard,I cannot log in; the system says my password i...,Sorry to hear you're having trouble accessing ...,Reset account credentials and confirmed succes...,36.53,0,very_negative,1,android,EU,2024-01-31 05:14:27
1,2024-10-20T06:15:49,individual,in_app,billing,security_concern,medium,closed_no_action,standard,I noticed a suspicious login on my account.,We take security very seriously. Our team is r...,Ticket closed without further action after no ...,238.32,0,neutral,3,web,other,2024-10-20 06:15:49
2,2023-08-27T16:08:33,enterprise,phone_transcript,login_auth,billing_problem,low,resolved,gold,My invoice amount is incorrect compared to the...,Thanks for reaching out about the billing issu...,Adjusted the invoice and issued a refund where...,61.32,0,very_negative,2,web,other,2023-08-27 16:08:33
3,2024-09-24T14:04:53,education,email,mobile_app,performance,low,closed_no_action,gold,Queries in the mobile app module are timing out.,We understand performance issues can be frustr...,Ticket closed without further action after no ...,226.93,0,positive,5,ios,other,2024-09-24 14:04:53
4,2024-11-01T16:14:51,education,in_app,billing,security_concern,urgent,resolved,gold,I noticed a suspicious login on my account.,We take security very seriously. This has been...,"Investigated logs, mitigated the issue, and up...",4.36,0,negative,3,api_client,APAC,2024-11-01 16:14:51


In [5]:
df['issue_type'].value_counts()  # Balanced data

issue_type
how_to              7711
account_access      7560
performance         7524
other               7488
feature_request     7480
billing_problem     7463
security_concern    7459
bug                 7428
Name: count, dtype: int64

In [6]:
df.shape

(60113, 18)

In [7]:
df = df[df['issue_type'] != 'other']

In [8]:
df.shape

(52625, 18)

In [9]:
num_classes = df['issue_type'].nunique()
num_classes

7

In [10]:
X = df['initial_message']
y = df['issue_type']

In [11]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.2,random_state=42,stratify=y)

In [12]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [13]:
X_train.shape

(42100,)

In [14]:
hub_layer = hub.KerasLayer("https://tfhub.dev/google/nnlm-en-dim128/2",
                           input_shape=[], dtype=tf.string, trainable=True)


# "https://tfhub.dev/google/universal-sentence-encoder/4"

In [15]:
model = tf.keras.Sequential([
    hub_layer,
    tf.keras.layers.Dense(32,activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

In [16]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 keras_layer (KerasLayer)    (None, 128)               124642688 
                                                                 
 dense (Dense)               (None, 32)                4128      
                                                                 
 dropout (Dropout)           (None, 32)                0         
                                                                 
 dense_1 (Dense)             (None, 7)                 231       
                                                                 
Total params: 124,647,047
Trainable params: 124,647,047
Non-trainable params: 0
_________________________________________________________________


In [17]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',     
    patience=1,              
    restore_best_weights=True
)


In [18]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [19]:
len(set(X_train).intersection(set(X_test)))

93

In [20]:
history = model.fit(
    X_train,      
    y_train,       
    validation_data=(X_test, y_test),  
    epochs=1,          
    batch_size=32,
    callbacks=[early_stop]
)


1316/1316 [==============================] - 2098s 2s/step - loss: 0.7068 - accuracy: 0.9186 - val_loss: 0.0345 - val_accuracy: 1.0000


In [21]:
model.save("my_model.keras")

In [22]:
model.evaluate(X_test, y_test)

329/329 [==============================] - 38s 116ms/step - loss: 0.0345 - accuracy: 1.0000


[0.03449515625834465, 1.0]

## Lets test model with unseen data 

In [23]:
test = pd.read_csv(r"D:\Intelligent-Support-Ticket\Project Implementation\Data\nan_tickets.csv")
test.head()

,index,created_at,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,platform,region
0,2,2024-06-18T21:35:54,small_business,chat,api_integration,bug,low,in_progress,standard,The api integration feature is not saving my c...,Thanks for reporting this bug. We will look in...,NaN,NaN,0,neutral,3,android,MEA
1,3,2025-12-25T15:59:52,education,chat,analytics_dashboard,account_access,medium,in_progress,standard,I cannot log in; the system says my password i...,Sorry to hear you're having trouble accessing ...,NaN,NaN,0,positive,5,android,LATAM
2,10,2023-04-02T12:27:06,individual,in_app,data_export,feature_request,medium,in_progress,standard,Our team needs more advanced options for data ...,Thanks for your feature suggestion. Our team i...,NaN,NaN,0,neutral,3,android,EU
3,12,2023-02-22T18:04:40,small_business,email,notifications,performance,medium,in_progress,standard,Overall performance has degraded in the last f...,We understand performance issues can be frustr...,NaN,NaN,0,positive,4,ios,other
4,14,2025-06-15T10:10:31,education,chat,mobile_app,feature_request,low,in_progress,standard,It would be great to have mobile app support i...,Thanks for your feature suggestion. We will lo...,NaN,NaN,0,neutral,0,api_client,MEA


In [24]:
test.shape

(39887, 18)

In [25]:
test.isna().sum()

index                        0
created_at                   0
customer_segment             0
channel                      0
product_area                 0
issue_type                   0
priority                     0
status                       0
sla_plan                     0
initial_message              0
agent_first_reply            0
resolution_summary       39887
resolution_time_hours    39887
reopened                     0
customer_sentiment           0
csat_score                   0
platform                     0
region                       0
dtype: int64

In [26]:
test = test[test['issue_type'] != 'other']

In [27]:
test.shape

(34912, 18)

In [28]:
X = test['initial_message']
y = encoder.transform(test['issue_type'])

In [29]:
model.evaluate(X,y)

1091/1091 [==============================] - 255s 234ms/step - loss: 0.0346 - accuracy: 1.0000


[0.034567929804325104, 1.0]

In [43]:
y_pred_prop = model.predict(["there is too much letancy in page response"])
y_pred = np.argmax(y_pred_prop,axis=1)
encoder.inverse_transform(y_pred)

1/1 [==============================] - 0s 127ms/step


array(['performance'], dtype=object)

In [44]:
import pickle

with open("issue_typeModel/label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)